In [1]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv")
x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)))

y=torch.from_numpy(numpy.nan_to_num(pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv")['sales'].to_numpy().astype(numpy.float32)))


In [5]:
train_x = torch.utils.data.DataLoader(x,batch_size=batch_size)
train_y = torch.utils.data.DataLoader(y,batch_size=batch_size)

In [6]:
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight.data, gain=0.02)
        
class RF_NET(nn.Module):
    def __init__(self):
        super().__init__()
        self.rafire = nn.Sequential(
            
            nn.Linear(9, 1524),
            nn.BatchNorm1d(1524),
            nn.ReLU(),
            
            nn.Linear(1524, 824),
            nn.BatchNorm1d(824),
            nn.ReLU(),
            
            nn.Linear(824, 424),
            nn.BatchNorm1d(424),
            nn.ReLU(),
            
            nn.Linear(424, 212),
            nn.BatchNorm1d(212),
            nn.ReLU(),
            
            nn.Linear(212, 124),
            nn.BatchNorm1d(124),
            nn.ReLU(),
            
            nn.Linear(124, 1)
        )

    def forward(self,x):
        #x,_=self.rafire_0(x)
        #print(x,x.shape)
        return self.rafire(x)

In [7]:
rf_net = RF_NET().type(torch.float32).to(device)
rf_net= nn.DataParallel(rf_net).to(device)
if (device.type == 'cuda') and (ngpu > 1):
    rf_net = nn.DataParallel(rf_net, list(range(ngpu)))
rf_net.apply(weights_init)

DataParallel(
  (module): RF_NET(
    (rafire): Sequential(
      (0): Linear(in_features=9, out_features=1524, bias=True)
      (1): BatchNorm1d(1524, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=1524, out_features=824, bias=True)
      (4): BatchNorm1d(824, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU()
      (6): Linear(in_features=824, out_features=424, bias=True)
      (7): BatchNorm1d(424, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU()
      (9): Linear(in_features=424, out_features=212, bias=True)
      (10): BatchNorm1d(212, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (11): ReLU()
      (12): Linear(in_features=212, out_features=124, bias=True)
      (13): BatchNorm1d(124, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (14): ReLU()
      (15): Linear(in_features=124, out_features=1, bias=True)
 

In [8]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(rf_net.parameters(), lr=lr,betas=(beta1, 0.999))
scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [9]:
high_rf,i,j_r,k,z_w_=100000,0,0,0,[]
if os.path.exists(f"/kaggle/working/jsr_rf")==False:
    os.mkdir(f"/kaggle/working/jsr_rf")
    
correct,total=0,0

while(True):
    optimizer.zero_grad()
    for data,label in zip(train_x,train_y):
        output = rf_net(data.to(device)).view(-1)
        err_real = criterion(output, label.to(device))
        err_real.backward()
        optimizer.step()

    print(f"EPOCH : {i} LOSS_wl : {err_real.item()}")

    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_,j_r=[],0
            print(f"lr_br:{optimizer.param_groups[0]['lr']}".upper())
            scheduler.step()
            print(f"lr_up:{optimizer.param_groups[0]['lr']}".upper())
            
    z_w_.append(err_real.item())
    if err_real.item()<high_rf:
            high_rf=err_real.item()
            torch.save(rf_net.state_dict(),f'/kaggle/working/jsr_rf/{err_real.item()}.pth')
    if i==10:
        break
    i+=1

/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/opt/conda/lib/python3.10/site-packages/torch/nn/modules/linear.py:117: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)


EPOCH : 0 LOSS_wl : 15.838799476623535
EPOCH : 1 LOSS_wl : 16.988052368164062
EPOCH : 2 LOSS_wl : 16.957256317138672
EPOCH : 3 LOSS_wl : 16.274450302124023
EPOCH : 4 LOSS_wl : 14.829777717590332
LR_BR:0.0001
LR_UP:8.6E-05
EPOCH : 5 LOSS_wl : 18.73616600036621
EPOCH : 6 LOSS_wl : 14.517993927001953
EPOCH : 7 LOSS_wl : 18.17695426940918
EPOCH : 8 LOSS_wl : 13.352540969848633
EPOCH : 9 LOSS_wl : 16.008773803710938
EPOCH : 10 LOSS_wl : 13.498077392578125
